# Visualisierung der Basisdaten

Die zusammengefuehrte Tabelle wird geladen. `slug`, `zeitpunkt` und `status` werden vor der Auswertung entfernt.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

datei = Path("../metadata/basicdata.csv")

data = pd.read_csv(datei, sep=";", encoding="utf-8-sig")
data = data.drop(columns=["slug", "zeitpunkt", "status"], errors="ignore")

data = data[data["Jahr"].isin([2024, 2025, 2026])]

data
data.to_csv("daten.csv", sep=";", index=False, encoding="utf-8")

In [ ]:
sns.set_theme(
    style="whitegrid",
    palette=["coral"],
    context="notebook"
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

fig.suptitle(
    "Überblick über die Kaggle-Datensätze",
    fontsize=16,
    fontweight="bold"
)

# 1. Fehlende Werte – oben links
sns.histplot(
    data=data, x="anteil_fehlend_prozent",
    bins=len(data), kde=True, ax=axes[0, 0]
)
axes[0, 0].set_title("Verteilung fehlender Werte")
axes[0, 0].set_xlabel("Fehlende Zellen (%)")

# 2. Aufgabentypen – oben rechts
sns.countplot(
    data=data, y="aufgabe_vermutet",
    order=data["aufgabe_vermutet"].value_counts().index,
    ax=axes[0, 1]
)
axes[0, 1].set_title("Verteilung der Aufgabentypen")
axes[0, 1].set_xlabel("Anzahl")
axes[0, 1].set_ylabel("Aufgabentyp")

# 3. Anzahl Features – unten links
sns.histplot(
    data=data, x="n_features",
    bins=len(data), kde=True, ax=axes[1, 0]
)
axes[1, 0].set_title("Verteilung der Feature-Anzahl")
axes[1, 0].set_xlabel("Anzahl Features")

# 4. Feature-Anzahl im Zeitverlauf – unten rechts
data_plot = data.dropna(subset=["n_features"]).copy()

data_plot["Zeit"] = (
    data_plot["Jahr"].astype(int).astype(str)
    + "-"
    + data_plot["Monat"].astype(int).astype(str).str.zfill(2)
)

sns.lineplot(
    data=data_plot,
    x="Zeit",
    y="n_features",
    marker="o",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Entwicklung der Feature-Anzahl")
axes[1, 1].set_xlabel("Veröffentlichungszeitpunkt")
axes[1, 1].set_ylabel("Anzahl Features")
axes[1, 1].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=data,
    x="Jahr",
    color="coral"
)

plt.title("Wettbewerbe pro Jahr", fontsize=14, fontweight="bold")
plt.xlabel("Jahr")
plt.ylabel("Anzahl Wettbewerbe")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

ax = sns.countplot(
    data=data,
    y="aufgabe_vermutet",
    order=data["aufgabe_vermutet"].value_counts().index,
    color="coral"
)

plt.title("Verteilung der Aufgabentypen", fontsize=14, fontweight="bold")
plt.xlabel("Anzahl Wettbewerbe")
plt.ylabel("Aufgabentyp")

multi_label = data[
    data["aufgabe_vermutet"].astype(str).str.contains(
        "multi_label", case=False, na=False
    )
]

if not multi_label.empty:
    ml_text = "Multi-Label-Datensätze: " + ", ".join(
        multi_label["Titel"].astype(str)
    )

    plt.figtext(
        0.35, 0.09,
        ml_text,
        ha="center",
        fontsize=9,
        color="#B05A5A",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor="#FFF0F0",
            edgecolor="coral"
        )
    )

plt.tight_layout(rect=[0, 0.10, 1, 1])
plt.show()

In [ ]:
min_features = data.loc[data["n_features"].idxmin()]
max_features = data.loc[data["n_features"].idxmax()]

plt.figure(figsize=(10, 6))

ax = sns.histplot(
    data=data,
    x="n_features",
    bins=len(data),
    kde=True,
    color="coral"
)

ax.axvline(
    min_features["n_features"],
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"Minimum: {min_features['Titel']}"
)

ax.axvline(
    max_features["n_features"],
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Maximum: {max_features['Titel']}"
)

plt.title("Verteilung der Feature-Anzahl", fontsize=14, fontweight="bold")
plt.xlabel("Anzahl Features")
plt.ylabel("Anzahl Wettbewerbe")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
max_missing = data.loc[data["anteil_fehlend_prozent"].idxmax()]

plt.figure(figsize=(10, 6))

ax = sns.histplot(
    data=data,
    x="anteil_fehlend_prozent",
    bins=len(data),
    kde=True,
    color="coral"
)

ax.axvline(
    max_missing["anteil_fehlend_prozent"],
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Maximum: {max_missing['Titel']}"
)

plt.title("Verteilung fehlender Werte", fontsize=14, fontweight="bold")
plt.xlabel("Fehlende Zellen (%)")
plt.ylabel("Anzahl Wettbewerbe")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

plt.figure(figsize=(12, 7))

# Kategorien und Farben festlegen
kategorien = data["aufgabe_vermutet"].dropna().unique()
farben = sns.color_palette("Set2", n_colors=len(kategorien))
farben_dict = dict(zip(kategorien, farben))

ax = sns.histplot(
    data=data,
    y="Metrik",
    hue="aufgabe_vermutet",
    multiple="stack",
    discrete=True,
    palette=farben_dict,
    legend=False
)

# Legende manuell erstellen
legenden_elemente = [
    Patch(facecolor=farben_dict[kategorie], label=kategorie)
    for kategorie in kategorien
]

ax.legend(
    handles=legenden_elemente,
    title="Aufgabentyp",
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.title("Evaluationsmetriken nach Aufgabentyp")
plt.xlabel("Anzahl Wettbewerbe")
plt.ylabel("Evaluationsmetrik")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=data,
    x="train_mb",
    bins=len(data),
    kde=True,
    color="coral"
)

plt.title("Verteilung der Trainingsdatensatzgrößen")
plt.xlabel("Dateigröße in MB")
plt.ylabel("Anzahl Wettbewerbe")
plt.tight_layout()
plt.show()